# Store Promotions - Bronze Ingestion & Sample Data Generation

Creates `uc1_raw_store_promotions` and populates it with synthetic promotion
records for each store, aligned to the same date range as
`uc1_raw_store_transactions` (2026-05-19 to 2026-07-20).

This bronze table feeds:
- `uc2_silver_dim_store` as a TERTIARY source (fan-in, joined on `store_id`)
- `uc2_silver_dim_store_promotions` (new standalone dim table)

In [0]:
dbutils.widgets.text("catalog", "learning_dev", "Catalog")
dbutils.widgets.text("schema", "s819534", "Schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

BRONZE_TABLE = f"{catalog}.{schema}.uc1_raw_store_promotions"

## 1. Bronze table DDL

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
  promo_id STRING,
  store_id STRING,
  promo_name STRING,
  promo_type STRING,
  start_date DATE,
  end_date DATE,
  discount_pct DOUBLE,
  business_date DATE,
  source_file_name STRING,
  ingestion_timestamp TIMESTAMP,
  _rescued_data STRING
)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'Synthetic store promotions bronze table feeding uc2_silver_dim_store (TERTIARY) and uc2_silver_dim_store_promotions.'
""")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType, DoubleType
import random
import datetime

PROMO_SCHEMA = StructType([
    StructField("promo_id", StringType(), False),
    StructField("store_id", StringType(), False),
    StructField("promo_name", StringType(), True),
    StructField("promo_type", StringType(), True),
    StructField("start_date", DateType(), False),
    StructField("end_date", DateType(), False),
    StructField("discount_pct", DoubleType(), True),
    StructField("business_date", DateType(), False),
    StructField("source_file_name", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("_rescued_data", StringType(), True),
])

PROMO_NAMES = [
    "Summer Blowout", "Weekend Special", "Clearance Event", "Loyalty Bonus",
    "Flash Sale", "Seasonal Discount", "Buy More Save More", "Holiday Promo",
]
PROMO_TYPES = ["PERCENT_OFF", "BOGO", "CLEARANCE", "LOYALTY"]

STORES = [f"ST{str(i).zfill(3)}" for i in range(1, 26)]
RANGE_START = datetime.date(2026, 5, 19)
RANGE_DAYS = 62

## 2. Synthetic record generator

In [0]:
def generate_promotion(seq, store_id, force_no_discount=False):
    start_offset = random.randint(0, RANGE_DAYS - 1)
    start_date = RANGE_START + datetime.timedelta(days=start_offset)
    duration_days = random.randint(2, 14)
    end_date = start_date + datetime.timedelta(days=duration_days)
    discount_pct = None if force_no_discount else round(random.uniform(5.0, 40.0), 1)

    return {
        "promo_id": f"PROMO-{seq:04d}",
        "store_id": store_id,
        "promo_name": f"{random.choice(PROMO_NAMES)} - {store_id}",
        "promo_type": random.choice(PROMO_TYPES),
        "start_date": start_date,
        "end_date": end_date,
        "discount_pct": discount_pct,
        "business_date": start_date,
        "source_file_name": "synthetic_gen_v1.csv",
        "ingestion_timestamp": datetime.datetime.utcnow(),
        "_rescued_data": None,
    }


def write_batch(records, description):
    rows = [
        (
            r["promo_id"], r["store_id"], r["promo_name"], r["promo_type"],
            r["start_date"], r["end_date"], r["discount_pct"], r["business_date"],
            r["source_file_name"], r["ingestion_timestamp"], r["_rescued_data"],
        )
        for r in records
    ]
    df = spark.createDataFrame(rows, schema=PROMO_SCHEMA)
    df.write.mode("append").partitionBy("business_date").saveAsTable(BRONZE_TABLE)
    print(f"Wrote {len(records)} records - {description}")

## Batch 1 - Clean promotions, 4 per store (100 records, all pass)

In [0]:
records = []
seq = 1
for store_id in STORES:
    for _ in range(4):
        records.append(generate_promotion(seq, store_id))
        seq += 1

write_batch(records, "Batch 1 - clean promotions, 4 per store")

## 3. Verification

In [0]:
display(spark.sql(f"""
SELECT store_id, COUNT(*) AS promo_count, ROUND(AVG(discount_pct), 2) AS avg_discount_pct
FROM {BRONZE_TABLE}
GROUP BY store_id
ORDER BY store_id
"""))